# APEC Project — Data Analysis

This notebook builds up, CTE by CTE, the two core queries that run in `personalization_impact.ipynb`: the shared pipeline (Part A) and `category_state_monthly_query`/`category_state_summary_query` (Part B). Each step below adds exactly one CTE and re-runs, so the final step in Part B reproduces the exact query and numbers used there.

For readability, every step here is scoped to a single illustrative month (April 2026) rather than the full 18-month window the real notebook uses — the query shape and every join are identical, just over a smaller slice of time so the intermediate row counts stay easy to look at.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)

MONTH_START = '2026-04-01'
MONTH_END = '2026-05-01'
ACTIVE_MAX_DAYS = 120
LAPSED_MAX_DAYS = 365
DORMANT_3YR_MIN_DAYS = 1095
SESSION_ATTRIBUTION_WINDOW_DAYS = 7

## Part A — The shared pipeline

`personalization_impact.ipynb`'s two queries are both built from the same four CTEs, just grouped differently at the end. Part A builds those four up one at a time.

### A1 — the `sends` CTE alone

Every Blueshift send in the month, no filtering by client or campaign type yet — just the date window and `holdout_group = 0` (excluding suppressed/holdout sends, which aren't part of the addressable campaign population).

In [2]:
query(f"""--sql
SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
FROM blueshift.campaign_activity_kpis
WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
  AND holdout_group = 0
ORDER BY client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,send_utm_campaign,send_utm_content
0,3,2026-04-19 22:15:58.111,email_us_k_static_fix_starsstripesandscute_041926_resend_up,email_us_k_static_fix_starsstripesandscute_041926_resend...
1,3,2026-04-18 14:54:25.549,email_us_w_dse_freestyle_centralized_aprildse_041826_active,email_us_w_dse_freestyle_centralized_aprildse_041826_act...
2,3,2026-04-26 18:40:25.324,email_us_n_transactional_trackeditem_outfordelivery,email_us_n_transactional_trackeditem_outfordelivery_1211...
3,3,2026-04-01 14:49:05.761,email_us_w_dse_freestyle_centralized_040126_active,email_us_w_dse_freestyle_centralized_marchdse_040126_act...
4,3,2026-04-24 22:03:08.194,email_us_n_survey_ai_042426,us_n_survey_ai_042426


In [3]:
query(f"""--sql
SELECT count(*) as n_sends, count(distinct client_id) as n_clients
FROM blueshift.campaign_activity_kpis
WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
  AND holdout_group = 0
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_sends,n_clients
0,249638382,7138407


249,638,382 sends across 7,138,407 distinct clients in April 2026 — this is StitchFix's entire send volume for the month, not yet restricted to any lifecycle state.

### A2 — add `eligible_state_sends`: the population filter

Join `sends` to `curated.checkout_based_client_state_journal` on `client_id`, with the send's own `sent_timestamp` falling inside a state period (`start_timestamp <= sent_timestamp < end_timestamp`) — this checks the client's lifecycle state **as of that specific send**, not their state today. Compute `lifecycle_state` from days-since-checkout (excluding `Never Active`), then keep only `Lapsed`, `Dormant`, and `Dormant 3+ yrs` — the three states `personalization_impact.ipynb` analyzes. `Active` clients are dropped here, same as `Never Active`.

In [4]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
)
SELECT
    s.client_id, s.sent_timestamp, j.last_buyable_checkout_ts,
    date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) as days_since_checkout,
    CASE
        WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
        WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
        WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
        ELSE 'Dormant 3+ yrs'
    END AS lifecycle_state
FROM sends s
INNER JOIN curated.checkout_based_client_state_journal j
    ON s.client_id = j.client_id
    AND s.sent_timestamp >= j.start_timestamp
    AND s.sent_timestamp <  j.end_timestamp
WHERE j.client_state_detail != 'Never Active'
ORDER BY s.client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,last_buyable_checkout_ts,days_since_checkout,lifecycle_state
0,3,2026-04-02 22:14:43.527,2025-09-29 03:12:08.796,185,Lapsed
1,3,2026-04-21 21:58:43.869,2025-09-29 03:12:08.796,204,Lapsed
2,3,2026-04-26 22:09:09.629,2025-09-29 03:12:08.796,209,Lapsed
3,3,2026-04-28 22:23:21.413,2025-09-29 03:12:08.796,211,Lapsed
4,3,2026-04-04 14:54:01.718,2025-09-29 03:12:08.796,187,Lapsed


In [5]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
)
SELECT lifecycle_state, count(*) as n_sends, count(distinct client_id) as n_clients
FROM state_sends
GROUP BY 1
ORDER BY n_clients DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,lifecycle_state,n_sends,n_clients
0,Active,96902927,1555807
1,Dormant 3+ yrs,38144005,1198458
2,Lapsed,32371636,741269
3,Dormant,25671028,739214


Of April 2026's 7.1M sent-to clients: 1.56M are Active (excluded), 1.20M are Dormant 3+ yrs, 741K are Lapsed, and 739K are Dormant — together, 2,613,906 distinct clients across the three states this analysis keeps (96,186,669 sends). `eligible_state_sends` in the real notebook is exactly this filtered to `lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')`.

### A3 — add `matched_sessions`: the session/UTM match

Join to `curated.user_session_conversion_metrics` on `client_id`, `utm_source = 'blueshift'`, and `utm_content` matching the send's own `send_utm_content` exactly, with the session landing within `SESSION_ATTRIBUTION_WINDOW_DAYS` of the send. This is the step that makes the metric **click-through-based**: a send only survives into this CTE if there's a real session carrying its UTMs.

In [6]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_content,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
),
eligible_state_sends AS (
    SELECT * FROM state_sends WHERE lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')
)
SELECT d.client_id, d.sent_timestamp, d.lifecycle_state, u.active_session_id, u.datetime_in_utc
FROM eligible_state_sends d
INNER JOIN curated.user_session_conversion_metrics u
    ON u.client_id = d.client_id
    AND u.utm_source = 'blueshift'
    AND u.utm_content = d.send_utm_content
    AND u.datetime_in_utc >= d.sent_timestamp
    AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
WHERE u.date_in_utc >= DATE '{MONTH_START}'
ORDER BY d.client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,lifecycle_state,active_session_id,datetime_in_utc
0,3,2026-04-22 16:06:48.040,Lapsed,61cf0734-d2b6-48c6-bb8e-ea229c8bcbf9,2026-04-22 16:19:25.808
1,3,2026-04-22 16:06:48.040,Lapsed,3e946482-2529-45ad-ad66-483aeaa9561a,2026-04-22 17:13:13.702
2,3,2026-04-28 17:09:02.965,Lapsed,ef64a8e4-0e11-4162-a82a-729e8d506f1b,2026-04-28 17:15:10.251
3,3,2026-04-28 17:09:02.965,Lapsed,5cb5a2e7-eba0-4cab-a513-6fb2e70fe231,2026-04-29 16:36:46.825
4,3,2026-04-24 18:15:58.647,Lapsed,dd924a0e-2729-4256-8dc9-16e2059f1353,2026-04-24 18:19:37.531


In [7]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_content,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
),
eligible_state_sends AS (
    SELECT * FROM state_sends WHERE lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')
)
SELECT count(distinct d.client_id) as n_clients_with_matched_session
FROM eligible_state_sends d
INNER JOIN curated.user_session_conversion_metrics u
    ON u.client_id = d.client_id
    AND u.utm_source = 'blueshift'
    AND u.utm_content = d.send_utm_content
    AND u.datetime_in_utc >= d.sent_timestamp
    AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
WHERE u.date_in_utc >= DATE '{MONTH_START}'
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_clients_with_matched_session
0,145544


145,544 of the 2,613,906 eligible clients (~5.6%) have any UTM-matched session at all this month — most sends never produce a trackable click-through, which is exactly why this metric reads as a lower bound on true reactivation.

### A4 — add `attributed_demand`: the demand confirmation

Join the matched session's `active_session_id` to `curated.client_reactivation_demand_events`, keeping only `demand_type IN ('fix', 'direct_buy')` — this confirms the session didn't just happen, it actually produced a real demand event.

In [8]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_content,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
),
eligible_state_sends AS (
    SELECT * FROM state_sends WHERE lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')
),
matched_sessions AS (
    SELECT d.client_id, d.sent_timestamp, d.lifecycle_state, u.active_session_id
    FROM eligible_state_sends d
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = d.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = d.send_utm_content
        AND u.datetime_in_utc >= d.sent_timestamp
        AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
)
SELECT DISTINCT ms.client_id, ms.sent_timestamp, ms.lifecycle_state, e.demand_id, e.demand_type, e.created_ts
FROM matched_sessions ms
INNER JOIN curated.client_reactivation_demand_events e
    ON e.active_session_id = ms.active_session_id
WHERE e.demand_type IN ('fix', 'direct_buy')
ORDER BY ms.client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,lifecycle_state,demand_id,demand_type,created_ts
0,3004024,2026-04-06 15:08:50.314,Dormant,665078531,fix,2026-04-06 21:05:23.246
1,3009975,2026-04-21 17:06:40.365,Lapsed,g8d6dbp2g,direct_buy,2026-04-21 17:07:58.221
2,3024686,2026-04-12 14:38:51.349,Lapsed,z2n7qp2kg,direct_buy,2026-04-13 05:06:41.741
3,3025345,2026-04-27 12:22:58.401,Dormant 3+ yrs,667900122,fix,2026-04-27 12:24:28.000
4,3025345,2026-04-27 12:22:58.401,Dormant 3+ yrs,667900123,fix,2026-04-27 12:24:28.000


In [9]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_content,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
),
eligible_state_sends AS (
    SELECT * FROM state_sends WHERE lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')
),
matched_sessions AS (
    SELECT d.client_id, d.sent_timestamp, u.active_session_id
    FROM eligible_state_sends d
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = d.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = d.send_utm_content
        AND u.datetime_in_utc >= d.sent_timestamp
        AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
),
attributed_demand AS (
    SELECT DISTINCT ms.client_id, ms.sent_timestamp
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events e
        ON e.active_session_id = ms.active_session_id
    WHERE e.demand_type IN ('fix', 'direct_buy')
)
SELECT count(*) as n_attributed, count(distinct client_id) as n_clients FROM attributed_demand
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_attributed,n_clients
0,11360,11130


11,130 clients confirmed reactivated this month, out of 145,544 with a matched session, out of 2,613,906 eligible. This is the numerator behind every reactivation rate in Part B, once split by category and state.

## Part B — `category_state_monthly_query` / `category_state_summary_query`

Same `sends` → `eligible_state_sends` → `matched_sessions` → `attributed_demand` mechanics as Part A. Two things are added: the full, ordered campaign-category mapping (instead of no categorization at all), and a `client_category_state` collapse that determines, per client per category per state, whether they reactivated at all.

### B1 — the full `categorized_sends` CTE alone

Every send gets exactly one category, checked in this order (first match wins): `APEC`, `TRANSACTIONAL`, `DSE`, `PROMO_INCENTIVE`, `FLS`, `TRACKING_ARTIFACT`, `PROMO_INCENTIVE` (the broader keyword scan), `WINBACK`, `BIRTHDAY`, `CROSSSELL`, else `OTHER`. This is a wider mapping than a simpler "APEC / TRANSACTIONAL / DSE / FLS / else OTHER" split — the extra categories matter because without them, everything they'd otherwise catch (one-time offer tests, a non-send tracking artifact, win-back-named campaigns, birthday sends, cross-sell) silently piles into `OTHER`, inflating it and making its reactivation rate harder to interpret.

In [10]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_campaign,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
),
eligible_state_sends AS (
    SELECT * FROM state_sends WHERE lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')
)
SELECT
    CASE
        WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC'
        WHEN send_utm_campaign LIKE '%browseabandon%' THEN 'APEC'
        WHEN send_utm_campaign LIKE '%stylepass%' THEN 'APEC'
        WHEN send_utm_campaign LIKE '%styleprofile%' THEN 'APEC'
        WHEN send_utm_campaign LIKE '%transactional%' THEN 'TRANSACTIONAL'
        WHEN regexp_like(send_utm_campaign, '(^|_)dse(_|$)') THEN 'DSE'
        WHEN send_utm_campaign LIKE '%tfy%' THEN 'DSE'
        WHEN send_utm_campaign LIKE '%cyl%' THEN 'DSE'
        WHEN send_utm_campaign LIKE '%offertest%' THEN 'PROMO_INCENTIVE'
        WHEN send_utm_campaign LIKE '%freestyle-lifecycle%' THEN 'FLS'
        WHEN send_utm_campaign LIKE '%freestyle_lifecycle%' THEN 'FLS'
        WHEN send_utm_campaign = 'crumbs_has_app' THEN 'TRACKING_ARTIFACT'
        WHEN send_utm_campaign LIKE '%offer%' OR send_utm_campaign LIKE '%discount%'
          OR send_utm_campaign LIKE '%promo%' OR send_utm_campaign LIKE '%sale%'
          OR send_utm_campaign LIKE '%percentoff%' OR send_utm_campaign LIKE '%markdown%'
          OR send_utm_campaign LIKE '%deal%' OR send_utm_campaign LIKE '%coupon%'
          OR send_utm_campaign LIKE '%spendget%' OR send_utm_campaign LIKE '%lastchance%'
          OR send_utm_campaign LIKE '%credit%' OR send_utm_campaign LIKE '%incentive%'
          OR send_utm_campaign LIKE '%waivedstylingfee%'
        THEN 'PROMO_INCENTIVE'
        WHEN send_utm_campaign LIKE '%reactivation%' OR send_utm_campaign LIKE '%winback%' THEN 'WINBACK'
        WHEN send_utm_campaign LIKE '%birthday%' THEN 'BIRTHDAY'
        WHEN send_utm_campaign LIKE '%crossell%' OR send_utm_campaign LIKE '%crosssell%' THEN 'CROSSSELL'
        ELSE 'OTHER'
    END AS campaign_category,
    count(*) as n_sends, count(distinct client_id) as n_clients
FROM eligible_state_sends
GROUP BY 1
ORDER BY n_clients DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,campaign_category,n_sends,n_clients
0,OTHER,25710076,2175442
1,PROMO_INCENTIVE,15457193,2110710
2,DSE,29178061,2023795
3,TRACKING_ARTIFACT,21508546,819003
4,APEC,2977625,425105
5,BIRTHDAY,299497,291676
6,TRANSACTIONAL,881935,251799
7,WINBACK,50638,49009
8,FLS,123098,41897


All ten categories show up in this single month. Notably, `PROMO_INCENTIVE` (2.11M clients) and `TRACKING_ARTIFACT` (819K) are each large enough that, if this mapping didn't exist, they'd have piled straight into `OTHER` — inflating it by roughly as much volume as `OTHER` has on its own (2.18M). This is the concrete reason the full mapping matters even though only `APEC`, `DSE`, and `OTHER` are profiled going forward: `OTHER` is only a clean, interpretable residual once everything else has somewhere else to go.

### B2 — the full `category_state_monthly_query` / `category_state_summary_query`

Add `matched_sessions` and `attributed_demand` from Part A, restrict to `campaign_category IN ('APEC', 'DSE', 'OTHER')`, then `client_category_state` (one row per client per category per state, `MAX()` of whether they reactivated), then the final aggregation. This is the exact query behind `personalization_impact.ipynb`'s full-window summary table, run here for a single month instead of the full window.

In [11]:
category_state_query = f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
state_sends AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_campaign, s.send_utm_content,
        CASE
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN 'Active'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
            WHEN date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
            ELSE 'Dormant 3+ yrs'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail != 'Never Active'
),
eligible_state_sends AS (
    SELECT * FROM state_sends WHERE lifecycle_state IN ('Lapsed', 'Dormant', 'Dormant 3+ yrs')
),
categorized_sends AS (
    SELECT
        client_id, sent_timestamp, send_utm_content, lifecycle_state,
        CASE
            WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%browseabandon%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%stylepass%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%styleprofile%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%transactional%' THEN 'TRANSACTIONAL'
            WHEN regexp_like(send_utm_campaign, '(^|_)dse(_|$)') THEN 'DSE'
            WHEN send_utm_campaign LIKE '%tfy%' THEN 'DSE'
            WHEN send_utm_campaign LIKE '%cyl%' THEN 'DSE'
            WHEN send_utm_campaign LIKE '%offertest%' THEN 'PROMO_INCENTIVE'
            WHEN send_utm_campaign LIKE '%freestyle-lifecycle%' THEN 'FLS'
            WHEN send_utm_campaign LIKE '%freestyle_lifecycle%' THEN 'FLS'
            WHEN send_utm_campaign = 'crumbs_has_app' THEN 'TRACKING_ARTIFACT'
            WHEN send_utm_campaign LIKE '%offer%' OR send_utm_campaign LIKE '%discount%'
              OR send_utm_campaign LIKE '%promo%' OR send_utm_campaign LIKE '%sale%'
              OR send_utm_campaign LIKE '%percentoff%' OR send_utm_campaign LIKE '%markdown%'
              OR send_utm_campaign LIKE '%deal%' OR send_utm_campaign LIKE '%coupon%'
              OR send_utm_campaign LIKE '%spendget%' OR send_utm_campaign LIKE '%lastchance%'
              OR send_utm_campaign LIKE '%credit%' OR send_utm_campaign LIKE '%incentive%'
              OR send_utm_campaign LIKE '%waivedstylingfee%'
            THEN 'PROMO_INCENTIVE'
            WHEN send_utm_campaign LIKE '%reactivation%' OR send_utm_campaign LIKE '%winback%' THEN 'WINBACK'
            WHEN send_utm_campaign LIKE '%birthday%' THEN 'BIRTHDAY'
            WHEN send_utm_campaign LIKE '%crossell%' OR send_utm_campaign LIKE '%crosssell%' THEN 'CROSSSELL'
            ELSE 'OTHER'
        END AS campaign_category
    FROM eligible_state_sends
),
focus_sends AS (
    SELECT * FROM categorized_sends WHERE campaign_category IN ('APEC', 'DSE', 'OTHER')
),
matched_sessions AS (
    SELECT c.client_id, c.sent_timestamp, c.lifecycle_state, c.campaign_category, u.active_session_id
    FROM focus_sends c
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = c.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = c.send_utm_content
        AND u.datetime_in_utc >= c.sent_timestamp
        AND u.datetime_in_utc <  c.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
),
attributed_demand AS (
    SELECT DISTINCT ms.client_id, ms.sent_timestamp, ms.lifecycle_state, ms.campaign_category
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events e
        ON e.active_session_id = ms.active_session_id
    WHERE e.demand_type IN ('fix', 'direct_buy')
),
client_category_state AS (
    SELECT
        d.client_id, d.lifecycle_state, d.campaign_category,
        MAX(CASE WHEN a.client_id IS NOT NULL THEN 1 ELSE 0 END) AS client_reactivated
    FROM focus_sends d
    LEFT JOIN attributed_demand a
        ON d.client_id = a.client_id AND d.sent_timestamp = a.sent_timestamp
        AND d.lifecycle_state = a.lifecycle_state AND d.campaign_category = a.campaign_category
    GROUP BY 1, 2, 3
)
SELECT
    campaign_category, lifecycle_state,
    COUNT(DISTINCT client_id) AS unique_clients_sent,
    SUM(client_reactivated) AS reactivated_clients,
    CAST(SUM(client_reactivated) AS DOUBLE) / COUNT(DISTINCT client_id) AS client_reactivation_rate
FROM client_category_state
GROUP BY 1, 2
ORDER BY 1, 2
"""

query(category_state_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,campaign_category,lifecycle_state,unique_clients_sent,reactivated_clients,client_reactivation_rate
0,APEC,Dormant,134333,57,0.000424
1,APEC,Dormant 3+ yrs,212419,39,0.000184
2,APEC,Lapsed,86126,56,0.000650
3,DSE,Dormant,566296,675,0.001192
4,DSE,Dormant 3+ yrs,989350,269,0.000272
5,DSE,Lapsed,515031,1357,0.002635
6,OTHER,Dormant,598014,592,0.000990
7,OTHER,Dormant 3+ yrs,1078354,271,0.000251
8,OTHER,Lapsed,546175,1006,0.001842


Even in a single month, the same pattern holds across all three focus categories: Lapsed clients reactivate at the highest rate, Dormant next, and Dormant 3+ yrs lowest — e.g. for OTHER, Lapsed is 0.18%, Dormant is 0.10%, Dormant 3+ yrs is 0.03%. This is the shape `personalization_impact.ipynb`'s full 18-month window quantifies more precisely, and it's the reason Lapsed/Dormant are included as context: the Dormant 3+ yrs population reactivates worst *no matter which of these three categories sends to it*, which matters for interpreting whether a weak APEC result there is about personalization specifically or about the population being harder to reach in general.